## Multiagent System

PS: BA gathers requirement from the client. Reverify (Human in the loop) the requirement with the client after it said its requirement.

If client agrees- passes to Manager.

If client desagress- update the requirement and reverify with client.

Manager takes the requirement and give this requirement to team lead. The team lead understand the requirement and creates a 1 sprint task. and assign it to the devlopers.

The developer starts working, once done , the code is given to QA.

If QA is satisfy, QA Passes the completion  msg to Team lead.

Team lead gives the update to Manager.

Manager updates the BA

BA updates the Client.

Client approves if he is satisfied.

If yes, Congrulations, Else the flow goes on again!



In [5]:
from typing import Annotated, Literal, Optional
from pathlib import Path
import sys
from IPython.display import display, Image
from dotenv import load_dotenv
from langchain.messages import SystemMessage
from langchain_openai import ChatOpenAI, data
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, StateGraph, add_messages
import sqlite3
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage  , HumanMessage,trim_messages ,RemoveMessage
from langgraph.checkpoint.sqlite import SqliteSaver
import json
from pydantic import BaseModel, Field
import os

In [6]:
llm=ChatOpenAI (
    model= 'gpt-4o-mini',
    temperature=0.3
)

In [ ]:
class GlobalState(BaseModel):
    requirement:str =Field(description="Requirement gathered by BA")
    client_query:str=Field(description="Clients  raw requirement")
    BA_MSG:str=Field(description="Remark of Buisness Analyst")
    Manager_MSG:str=Field(description="Remark of Manager")
    Teamlead_MSG:str=Field(description="Remark of Team Lead")
    QA_MSG:str=Field(description="Remark of Quality Assurance staff")
    Developer_MSG:str=Field(description="Remark of Developers")

    current_stage: Literal['BA','MANAGER','TEAM_LEAD','QA','DEVELOPER'] = "BA"
    is_completed : bool = False
    is_requirement_clear: bool = False

In [16]:
# Making Agents Node


def BA(State:GlobalState,BA_response):
    ba_prompt = ChatPromptTemplate.from_template('''
        You are a Business Analyst. Analyze the requirement: 'client_requirement'

        If the requirement is unclear or missing key details, output: 
        [STATUS: CLARIFY] followed by exactly 3 specific questions for the client.

        If the requirement is clear and actionable, output: 
        [STATUS: PASSED] followed by a concise summary for the Manager.

        ### client_requirement
        {client_requirement}                                         
''')
    res=ba_prompt | llm
    content=res.invoke({"client_requirement" : State.client_query })

    return {
        State.requirement:content.content,
        State.is_requirement_clear:True,
        State.current_stage:"MANAGER"
    }
